In [1]:
import numpy as np 
import torch
import torch.nn as nn

np.random.seed(0)   # NumPy
torch.manual_seed(0)  # PyTorch

In [57]:
rng = np.random.default_rng(42)

## Regression Losses

In [61]:
regr_y_true = rng.uniform(0,10, size =(20,))
print(regr_y_true)
regr_y_pred = rng.uniform(0,10, size = regr_y_true.shape)
print(regr_y_pred)

[6.6431354  4.06386861 8.14020385 1.6697292  0.22712073 0.90047861
 7.22359351 4.6187723  1.61271779 5.01044775 1.52312103 6.96320375
 4.46156276 3.81021226 3.01512089 6.30282593 3.61812611 0.87649919
 1.18005902 9.61897665]
[9.08580691 6.99707134 2.65869961 9.69176377 7.78750904 7.16890189
 4.49361502 2.72241562 0.96390962 9.02602397 4.5577629  2.02363365
 3.05956624 5.79219569 1.76772783 8.56614284 7.5851953  7.19462956
 4.3209304  6.27308841]


### Mean Squared Errors

In [62]:
def mean_square_err(y_true:np.array, y_pred:np.array) -> np.float32:
    N = len(y_pred)
    return np.mean((y_pred - y_true)**2)

def derivative_mean_square_err(y_true:np.array, y_pred:np.array) -> np.float32:
    N = len(y_pred)
    return np.mean(2*(y_pred - y_true))


def mean_square_torch(y_true:torch.tensor, y_pred:torch.tensor):
    loss= nn.MSELoss()
    return loss(y_true, y_pred)


In [66]:

loss_1 = mean_square_err(regr_y_true, regr_y_pred) 
loss_2 = mean_square_torch(torch.tensor(regr_y_true), torch.tensor(regr_y_pred))
print(loss_1, loss_2)
assert np.isclose(loss_1, loss_2.item())

17.795745349382265 tensor(17.7957, dtype=torch.float64)


### Mean Absolute Errors

In [67]:
def mean_abs_err(y_true:np.array, y_pred:np.array) -> np.float32:
    N = len(y_pred)
    return np.mean(np.abs(y_pred - y_true))

def derivative_mean_abs_err(y_true:np.array, y_pred:np.array) -> np.float32:
    N = len(y_pred)
    return np.where((y_pred - y_true) < 0, -1, 1)

def mean_abs_torch(y_true:torch.tensor, y_pred:torch.tensor):
    loss= nn.L1Loss()
    return loss(y_true, y_pred)

In [68]:
loss_1 = mean_abs_err(regr_y_true, regr_y_pred) 
loss_2 = mean_abs_torch(torch.tensor(regr_y_true), torch.tensor(regr_y_pred))
print(loss_1, loss_2)
assert np.isclose(loss_1, loss_2.item())

3.6819902620321896 tensor(3.6820, dtype=torch.float64)


## Classification Losses

### BinaryCrossEntropy

In [99]:
class_y_true = rng.uniform(0, 1, size=(20,)).astype(np.float32)

print(class_y_true)
class_logits = rng.normal(0, 1, size=(20,)).astype(np.float32)
print(class_y_pred)

[0.4087831  0.21762852 0.58830625 0.31704092 0.03605983 0.41840005
 0.4741327  0.22559287 0.5724579  0.5657719  0.70200217 0.6479485
 0.65243304 0.31621414 0.7874322  0.5491444  0.43141818 0.6260125
 0.36065733 0.51273924]
[0.57241315 0.14513025 0.94602445 0.30134263 0.57801722 0.69977594
 0.64923316 0.94059441 0.14843899 0.50835274 0.40403439 0.47416873
 0.11921753 0.13409461 0.27807555 0.3047046  0.42790321 0.61098755
 0.63462912 0.4118109 ]


In [108]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def binary_cross_entropy_loss(y_true, logits):
    y_prob= np.clip(sigmoid(logits), 1e-7, 1 - 1e-7)
    return np.mean(-1*(y_true*np.log(y_prob) + (1-y_true)*np.log(1-y_prob)))

def derivative_binary_cross_entropy_loss_torch(y_true, y_pred):
    y_prob =sigmoid(logits)
    return y_prob-y_true

def binary_cross_entropy_loss_torch(y_true, logits):
    loss = nn.BCEWithLogitsLoss()
    return loss(
        torch.tensor(logits, dtype=torch.float32),
        torch.tensor(y_true, dtype=torch.float32)
    )

In [109]:
loss_1= binary_cross_entropy_loss(class_y_true,class_logits)
loss_2= binary_cross_entropy_loss_torch(class_y_true, class_logits)

print(loss_1, loss_2)
np.isclose(loss_1, loss_2)


0.7630629 tensor(0.7631)


np.True_

### Softmax

In [110]:
class_y_true_cat = rng.uniform(0, 7, size=(20,)).astype(np.float32)

print(class_y_true)
class_logits_cat = rng.normal(0, 7, size=(20,)).astype(np.float32)
print(class_y_pred)

[0.4087831  0.21762852 0.58830625 0.31704092 0.03605983 0.41840005
 0.4741327  0.22559287 0.5724579  0.5657719  0.70200217 0.6479485
 0.65243304 0.31621414 0.7874322  0.5491444  0.43141818 0.6260125
 0.36065733 0.51273924]
[0.57241315 0.14513025 0.94602445 0.30134263 0.57801722 0.69977594
 0.64923316 0.94059441 0.14843899 0.50835274 0.40403439 0.47416873
 0.11921753 0.13409461 0.27807555 0.3047046  0.42790321 0.61098755
 0.63462912 0.4118109 ]


In [ ]:
def softmax(x):
    exp_ex = np.exp(x - np.max(x, axis=1, keepdim=True))
    return exp_ex /np.sum(exp_x, axis=1, keepdim=True)

def categorical_cross_entropy_loss(y_true, logits):
    y_prob= softmax(logits)
    y_prob = np.clip(y_prob, 1e-7, 1 - 1e-7)
    return -1* np.mean(np.log(y_prov))

def derivative_binary_cross_entropy_loss_torch(y_true, y_pred):
    y_prob =sigmoid(logits)
    return y_prob-y_true

def binary_cross_entropy_loss_torch(y_true, logits):
    loss = nn.CrossEntropyLoss()
    return loss(
        torch.tensor(logits, dtype=torch.float32),
        torch.tensor(y_true, dtype=torch.float32)
    )